**Task:** Given an NL question and a database schema, identify the tables and columns the underlying SQL would reference, outputting structured JSON:

```json
{"TableName1": ["Col1", "Col2"], "TableName2": ["Col3"]}
```

**Reference notebook:** `rf-tutorial-sft-chatqa-lite.ipynb`  
**Models (≤2B, instruction-tuned):** Qwen2.5-0.5B-Instruct, Qwen2.5-1.5B-Instruct  
**Framework:** RapidFire AI multi-config SFT (wraps HuggingFace TRL SFTTrainer + PEFT)

## 0. Install Dependencies

In [ ]:
# Run once on a fresh DSMLP pod
# !pip install -q rapidfireai transformers datasets peft trl accelerate bitsandbytes
# !pip install -q evaluate rouge_score

To check if all packages are installed, run the following:

In [ ]:
packages = [
    "rapidfireai", "transformers", "datasets", "peft", "trl",
    "accelerate", "bitsandbytes", "evaluate", "rouge_score",
]

for pkg in packages:
    try:
        mod = __import__(pkg.replace("-", "_").split("[")[0])
        ver = getattr(mod, "__version__", "?")
        print(f"OK       {pkg:20} {ver}")
    except ImportError:
        print(f"MISSING  {pkg}")

## 1. Imports

In [ ]:
import json
from functools import partial
from pathlib import Path
from typing import Callable
from paths import TRAIN_JSON, VAL_INPUT_JSON, VAL_JSON, require_release_data, setup

# RapidFire AI — mirrors the tutorial import exactly
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

from datasets import Dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Load Training & Validation Data

The release packet provides `train.json` (301 examples) and `validation.json` (101 examples). Schemas live in `schemas/` as Spider-format JSON files, one per db.

In [ ]:
setup(chdir_try=True)
require_release_data()

from schema_formatting import (
    format_row,
    make_formatting_func,
    sort_schema_links,
)
from schema_metrics import schema_link_metrics
from schema_metrics import compute_metrics_for_rf

TRAIN_PATH = TRAIN_JSON
VAL_PATH = VAL_JSON
VAL_INPUT_PATH = VAL_INPUT_JSON

with open(TRAIN_PATH) as f:
    raw_train = json.load(f)
with open(VAL_PATH) as f:
    raw_val = json.load(f)
with open(VAL_INPUT_PATH) as f:
    raw_val_input = json.load(f)

# train = raw_train
# val = raw_val

# train_ds = Dataset.from_list(train)
# val_ds = Dataset.from_list(val)

# fmt = make_formatting_func("minimal")

# print(f"Train: {len(train)} | Tiny val: {len(val)}")
# print("Train keys:", list(train[0].keys()))

# Small subsets for fast iteration
# N_TRAIN, N_VAL = 64, 32
# tiny_train = raw_train[:N_TRAIN]
# tiny_val = raw_val[:N_VAL]

# train_ds = Dataset.from_list(tiny_train)
# val_ds = Dataset.from_list(tiny_val)

# fmt = make_formatting_func("minimal")

# print(f"Tiny train: {len(tiny_train)} | Tiny val: {len(tiny_val)}")
# print("Train keys:", list(tiny_train[0].keys()))

When you're ready, train on the real thing!

In [ ]:
train = raw_train
val = raw_val

train_ds = Dataset.from_list(train)
val_ds = Dataset.from_list(val)

fmt = make_formatting_func("minimal")

print(f"Train: {len(train)} | Tiny val: {len(val)}")
print("Train keys:", list(train[0].keys()))

In [ ]:
import gc

# resolving errors when there are failed experiments
def cuda_cleanup():
  """Call between failed runs to reduce allocator/NVML pressure."""
  gc.collect()
  try:
    import torch
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.synchronize()
      print("CUDA cache cleared")
  except Exception as e:
    print(f"cuda_cleanup skipped: {e}")

## Before Proceeding: for steps 3 through 6 
Shows how the helper files `schema_formatting` and `schema_metrics work`.
Skip to 7 for actual experimenation.

## 3. Schema Serialization Helpers

Two serialization strategies are compared in the multi-config sweep:
- **`minimal`** — table names + column names only (short context, lower token count)
- **`full`** — adds column types, primary-key flags, and foreign-key pairs (richer inductive bias), can use this for optional ablation material.

The serialization format directly affects sequence length, so it is a meaningful training knob.

In [ ]:
_DEFAULT_REPO_ROOT = Path.cwd().parent   # cwd is try/ after setup()
DEFAULT_SCHEMAS_DIR = _DEFAULT_REPO_ROOT / "schemas"
_schema_cache: dict = {}

def load_schema(db_id: str, schemas_dir: Path | str = DEFAULT_SCHEMAS_DIR) -> dict:
    """Load Spider-format schema JSON for db_id (spaces → underscores in filename)."""
    schemas_dir = Path(schemas_dir)
    if db_id not in _schema_cache:
        fname = db_id.replace(" ", "_") + ".json"
        with open(schemas_dir / fname) as f:
            _schema_cache[db_id] = json.load(f)
    return _schema_cache[db_id]
_sample_schema = load_schema(raw_train[0]["db_id"]) # expect NTSB, first one on train.json
print(_sample_schema)

In [ ]:
def serialize_schema_minimal(schema: dict) -> str:
    """Table names + column names only — short prompt, low token cost."""
    lines = []
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]   # list of [table_idx, col_name]
    for t_idx, tname in enumerate(table_names):
        cols = [c for (ti, c) in column_names if ti == t_idx]
        lines.append(f"{tname}({', '.join(cols)})")
    return "\n".join(lines)

print(serialize_schema_minimal(_sample_schema)) # should expect only table and column names

In [ ]:
def serialize_schema_full(schema: dict) -> str:
    """Adds column types, PK flags, FK pairs — richer context, longer prompt."""
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]
    column_types = schema["column_types"]
    primary_keys = set(schema["primary_keys"])      # column indices
    foreign_keys = schema.get("foreign_keys", [])        # [[from_col_idx, to_col_idx], ...]

    # Build FK lookup: col_idx -> referenced "Table.Column"
    fk_lookup = {}
    for (src, tgt) in foreign_keys:
        tgt_table = table_names[column_names[tgt][0]]
        tgt_col   = column_names[tgt][1]
        fk_lookup[src] = f"{tgt_table}.{tgt_col}"

    lines = []
    for t_idx, tname in enumerate(table_names):
        col_parts = []
        for c_idx, (ti, cname) in enumerate(column_names):
            if ti != t_idx:
                continue
            ctype  = column_types[c_idx] if c_idx < len(column_types) else "text"
            flags  = []
            if c_idx in primary_keys:
                flags.append("PK")
            if c_idx in fk_lookup:
                flags.append(f"FK->{fk_lookup[c_idx]}")
            # flag_str = f" [{', '.join(flags)}]" if flags else
            flag_str = " [" + ", ".join(flags) + "]" if flags else ""
            col_parts.append(f"{cname} {ctype}{flag_str}")
        lines.append(f"{tname}(\n  " + ",\n  ".join(col_parts) + "\n)")
    return "\n".join(lines)


print(serialize_schema_full(_sample_schema)) # should expect table, column names, and info about the columns

## 4. Data Formatting Functions

Following the tutorial's `sample_formatting_function` pattern, we define two formatters (one per schema style) that return `{prompt: [...], completion: [...]}` dicts in ChatML multi-turn format consumed by `RFSFTConfig`.

**Design notes:**
- The system prompt instructs the model to emit *only* a JSON object — no prose, no markdown fences.
- Schema links in the target completion are **sorted** (tables alphabetically, columns within table)   for deterministic decoding, reducing output variance for a fixed-capacity SLM.
- Identifiers are kept in their original casing from the schema file to avoid casing confusion at decode time.

In [ ]:
SYSTEM_PROMPT = (
    "You are a schema-linking assistant. Given a natural-language question and a database schema, "
    "output ONLY a JSON object mapping each referenced table name to the list of referenced column names. "
    "If a table is referenced but no specific columns are needed (e.g. SELECT COUNT(*)), map it to []. "
    "Do not output any explanation, markdown, or extra text — only the JSON object."
)

In [ ]:
# def _sort_schema_links(schema_links: dict) -> dict:
#     """Sort tables and columns alphabetically for stable decoding."""
#     return {t: sorted(cols) for t, cols in sorted(schema_links.items())}

# print(_sort_schema_links(_sample_schema))  # should be sorting columns alphabetically in NTSB.json

# new vesion for errors:
def sort_schema_links(schema_links: dict | None) -> dict:
    """Sort tables/columns; tolerate None column lists."""
    if schema_links is None:
        raise TypeError("schema_links is None — row has no gold labels (wrong dataset or double-format?)")
    if not isinstance(schema_links, dict):
        raise TypeError(f"schema_links must be dict, got {type(schema_links)}")
    out = {}
    for table, cols in sorted(schema_links.items()):
        out[table] = sorted(cols) if cols is not None else []
    return out

print(sort_schema_links(_sample_schema))  # should be sorting columns alphabetically in NTSB.json

In [ ]:
def format_row(row: dict, schema_style: str = "minimal") -> dict:
    """
    Convert one train/val example to {prompt, completion} chat format.

    Idempotent: if row is already formatted (has prompt + completion), returns unchanged.
    Safe for RapidFire calling map(formatting_func) more than once.
    """
    if "prompt" in row and "completion" in row:
        return {"prompt": row["prompt"], "completion": row["completion"]}

    links = row.get("schema_links")
    if links is None:
        raise ValueError(
            "schema_links missing on row. "
            f"keys={list(row.keys())} question_id={row.get('question_id')}. "
            "Use validation.json (not validation_input.json) for training eval."
        )

    serialize = serialize_schema_minimal if schema_style == "minimal" else serialize_schema_full
    schema_s = serialize(load_schema(row["db_id"]))
    user_msg = f"Database schema:\n{schema_s}\n\nQuestion: {row['question']}"
    target = json.dumps(sort_schema_links(links), ensure_ascii=False)

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        "completion": [
            {"role": "assistant", "content": target},
        ],
    }
def make_formatting_func(schema_style: str = "minimal") -> Callable[[dict], dict]:
    """Picklable formatter for RFModelConfig.formatting_func."""
    return partial(format_row, schema_style=schema_style)

In [ ]:
# defining some functions
format_minimal = make_formatting_func("minimal")
format_full = make_formatting_func("full")


"""
Follows the Validation Input db_id and question. Expect for one training example,
 
Database schema:
Table INJURY: AIS, REGION, ...
Table CRASH: CASEID, ...

Question: Show a count of injuries by body region where...

System Message has the role : "Output only JSON ..."
User message has the Schema and NL
Assisant Message gets the schema_links from train to learn the gold answers
"""
sample = format_minimal(raw_train[0]) # expect the 
print("SYSTEM:", sample["prompt"][0]["content"])
print("USER:",   sample["prompt"][1]["content"])
print("ASSISTANT:", sample["completion"][0]["content"])

In [ ]:
# old format, seems to duplicate format when not needed
# def make_formatting_func(schema_style: str = "minimal"):
#     """Factory: returns a formatting function for the given schema serialization style."""
#     serialize = serialize_schema_minimal if schema_style == "minimal" else serialize_schema_full

#     def formatting_func(row):
#         schema = load_schema(row["db_id"])
#         schema_s = serialize(schema) # either min or full
#         user_msg = (
#             f"Database schema:\n{schema_s}\n\n"
#             f"Question: {row['question']}"
#         )
#         target = json.dumps(_sort_schema_links(row["schema_links"]), ensure_ascii=False)
#         return {
#             "prompt": [
#                 {"role": "system","content": SYSTEM_PROMPT},
#                 {"role": "user", "content": user_msg},
#             ],
#             "completion": [
#                 {"role": "assistant", "content": target} # where the SFT trains
#             ]
#         }
#     return formatting_func


# # defining some functions
# format_minimal = make_formatting_func("minimal")
# format_full = make_formatting_func("full")


# """
# Follows the Validation Input db_id and question. Expect for one training example,
 
# Database schema:
# Table INJURY: AIS, REGION, ...
# Table CRASH: CASEID, ...

# Question: Show a count of injuries by body region where...

# System Message has the role : "Output only JSON ..."
# User message has the Schema and NL
# Assisant Message gets the schema_links from train to learn the gold answers
# """
# sample = format_minimal(raw_train[0]) # expect the 
# print("SYSTEM:", sample["prompt"][0]["content"])
# print("USER:",   sample["prompt"][1]["content"])
# print("ASSISTANT:", sample["completion"][0]["content"])

## 5. Build HuggingFace Datasets

We wrap the raw JSON lists into `datasets.Dataset` objects. The formatting functions are applied inside `RFSFTConfig` (same pattern as the tutorial), so the dataset just needs to carry the raw columns.

In [ ]:
train_dataset = Dataset.from_list(raw_train)
eval_dataset  = Dataset.from_list(raw_val)

# Optional: shuffle training set for better mini-batch diversity
train_dataset = train_dataset.shuffle(seed=42)

print(f"Train: {len(train_dataset)} | Val: {len(eval_dataset)}")
print("Columns:", train_dataset.column_names)

## 6. Custom Eval Metrics — Schema-Link P/R/F1

Unlike the tutorial's ROUGE/BLEU metrics, schema linking requires **set-based** identifier matching. We implement the same scoring logic used by `eval.py` in the release packet:

- **Table P/R/F1** — over the set of table names
- **Column P/R/F1** — over the set of `Table.Column` pairs
- **Leaderboard score** = 0.5 × (PT+RT+F1T)/3 + 0.5 × (PC+RC+F1C)/3

Identifier comparisons are case-insensitive. Identifiers absent from the schema count as false positives (hallucinations), directly penalising precision.

In [ ]:
import re, json as _json

def _parse_json_output(text: str) -> dict:
    """Robustly parse model output: strip markdown fences, extract first {...} block."""
    text = text.strip()
    # Strip ```json ... ``` fences
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$",         "", text)
    # Extract first JSON object
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return _json.loads(m.group())
        except _json.JSONDecodeError:
            pass
    return {}


# from the project 2 statement
def _prf1(pred_set: set, gold_set: set):
    """Compute precision, recall, F1 for two sets."""
    if not gold_set and not pred_set:
        return 1.0, 1.0, 1.0
    if not gold_set:                      # empty-gold edge case
        return 0.0, 1.0, 0.0
    if not pred_set:
        return 0.0, 0.0, 0.0
    tp  = len(pred_set & gold_set)
    p   = tp / len(pred_set)
    r   = tp / len(gold_set)
    f1  = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


def schema_link_metrics(predictions: list[str], references: list[str]) -> dict:
    """
    Compute macro-averaged table and column P/R/F1.
    Both predictions and references are JSON strings mapping table->list[col].
    Mirrors the logic in the release packet eval.py.
    """
    t_p_list, t_r_list, t_f1_list = [], [], []
    c_p_list, c_r_list, c_f1_list = [], [], []

    for pred_str, gold_str in zip(predictions, references):
        pred_links = _parse_json_output(pred_str)
        gold_links = _json.loads(gold_str) if isinstance(gold_str, str) else gold_str

        # ---- Table-level ----
        pred_tables = {t.lower() for t in pred_links}
        gold_tables = {t.lower() for t in gold_links}
        tp, tr, tf = _prf1(pred_tables, gold_tables)
        t_p_list.append(tp); t_r_list.append(tr); t_f1_list.append(tf)

        # ---- Column-level (table.column pairs) ----
        pred_cols = {
            f"{t.lower()}.{c.lower()}"
            for t, cols in pred_links.items()
            for c in (cols or [])
        }
        gold_cols = {
            f"{t.lower()}.{c.lower()}"
            for t, cols in gold_links.items()
            for c in (cols or [])
        }
        cp, cr, cf = _prf1(pred_cols, gold_cols)
        c_p_list.append(cp); c_r_list.append(cr); c_f1_list.append(cf)

    def _mean(lst): return round(sum(lst) / len(lst), 4) if lst else 0.0

    PT, RT, F1T = _mean(t_p_list), _mean(t_r_list), _mean(t_f1_list)
    PC, RC, F1C = _mean(c_p_list), _mean(c_r_list), _mean(c_f1_list)
    table_score  = (PT + RT + F1T) / 3
    col_score    = (PC + RC + F1C) / 3
    leaderboard  = round(0.5 * table_score + 0.5 * col_score, 4)

    return {
        "table_P": PT, "table_R": RT, "table_F1": F1T,
        "col_P":   PC, "col_R":   RC, "col_F1":   F1C,
        "leaderboard_score": leaderboard,
    }


def compute_metrics_for_rf(eval_preds):
    """Adapter matching the signature expected by RFSFTConfig / TRL's compute_metrics."""
    predictions, labels = eval_preds
    # predictions and labels arrive as strings (decoded by RF trainer)
    return schema_link_metrics(list(predictions), list(labels))


print("Metrics function ready. Quick test:")
test_gold = '{"INJURY": ["AIS", "REGION"]}'
test_pred = test_gold   # perfect match
print(schema_link_metrics([test_pred], [test_gold]))  # expect 1
test_pred = '{"INJURY": ["AIS"]}'  # partial match
print(schema_link_metrics([test_pred], [test_gold])) # expect less than 1

## 7. Model Creation Function

Mirrors `sample_create_model` from the tutorial. Handles both standard `float16` loading (LoRA) and 4-bit quantised loading (QLoRA) depending on `model_kwargs`.

In [ ]:
def create_model(model_config: dict):
    """
    Create (model, tokenizer) for any config in the sweep.
    Supports LoRA (fp16/bf16) and QLoRA (4-bit NF4) via model_kwargs.
    Must return a (model, tokenizer) tuple — identical contract to the tutorial.
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer # added for the workers to work

    model_name   = model_config["model_name"]
    model_kwargs = model_config["model_kwargs"].copy()

    use_qlora = model_kwargs.pop("use_qlora", True) # Reduce memory!

    if use_qlora:
        from transformers import BitsAndBytesConfig
        import torch
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_cfg, **model_kwargs
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # Ensure a pad token exists (required by TRL's SFTTrainer)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id

    return (model, tokenizer)

## 8. Multi-Config Sweep — RFGridSearch

We define **two config axes** starting off some meaningful configurations.

| Axis | Values |
|------|--------|
| Base model | `Qwen2.5-0.5B-Instruct`, `Qwen2.5-1.5B-Instruct` |
| LoRA adapter | rank-8/α-16 (narrow), rank-32/α-64 (wide) |
| Learning rate | `1e-4`, `2e-4` |

This gives 2 × 2 × 2 = **8 runs**. Additionally, **schema serialization** (minimal vs. full) can be toggled by swapping `formatting_func`, giving a natural follow-up experiment axis once the baseline is stable.

**Tip:** Start with the 0.5B model + rank-8 + 1e-4 config to validate the pipeline cheaply, then scale up.

In [ ]:
# ── LoRA adapters ──────────────────────────────────────────────────────────
peft_narrow = RFLoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],           # minimal target modules
    bias="none"
)

peft_wide = RFLoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # all attention projections
    bias="none"
)

peft_configs = List([peft_narrow, peft_wide])

# ── SFT training args (two LR values × two models = 4 combos each) ─────────
# def make_sft_args(lr: float) -> RFSFTConfig:
#     return RFSFTConfig(
#         learning_rate=lr,
#         lr_scheduler_type="cosine",
#         per_device_train_batch_size=4,
#         per_device_eval_batch_size=4,
#         gradient_accumulation_steps=4,   # effective batch = 16 on a single MIG slice
#         num_train_epochs=3,
#         max_length=1024,             # cap to avoid OOM on long schema prompts
#         logging_steps=10,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         load_best_model_at_end=True,
#         metric_for_best_model="leaderboard_score",
#         greater_is_better=True,
#         bf16=True,                       # Blackwell supports BF16
#         fp16=False,
#     )

# made this to be compatible with RFAI
def make_sft_args(lr: float) -> RFSFTConfig:
    return RFSFTConfig(
        learning_rate=lr,
        lr_scheduler_type="cosine",
        per_device_train_batch_size=2, # changed this to 2! consumed too much memory
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        max_length=1024,
        logging_steps=10,
        eval_strategy="epoch",
        load_best_model_at_end=False,   # required with RapidFire
        bf16=True,
        fp16=False,
        report_to="none",
    )

# ── Schema style to formatting function mapping ────────────────────────────
# Change "minimal" -> "full" here to run the schema-style ablation
SCHEMA_STYLE = "minimal"   # or "full"
fmt_func = make_formatting_func(SCHEMA_STYLE)

# ── Config grid: 2 models × 2 LoRA × 2 LR = 8 configs ────────────────────
config_set = List([
    # ---- Qwen2.5-0.5B-Instruct ----
    RFModelConfig(
        model_name="Qwen/Qwen2.5-0.5B-Instruct",
        peft_config=peft_configs,           # List → grid will expand to 2 sub-configs
        training_args=make_sft_args(1e-4),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=fmt_func,
        compute_metrics=compute_metrics_for_rf,
        generation_config={
            "max_new_tokens": 256,
            "do_sample": False,           # greedy for structured output — more deterministic
            "temperature": 1.0,
        }
    ),
    RFModelConfig(
        model_name="Qwen/Qwen2.5-0.5B-Instruct",
        peft_config=peft_configs,
        training_args=make_sft_args(2e-4),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=fmt_func,
        compute_metrics=compute_metrics_for_rf,
        generation_config={
            "max_new_tokens": 256,
            "do_sample": False,
            "temperature": 1.0,
        }
    ),
    # ---- Qwen2.5-1.5B-Instruct ----
    RFModelConfig(
        model_name="Qwen/Qwen2.5-1.5B-Instruct",
        peft_config=peft_configs,
        training_args=make_sft_args(1e-4),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=fmt_func,
        compute_metrics=compute_metrics_for_rf,
        generation_config={
            "max_new_tokens": 256,
            "do_sample": False,
            "temperature": 1.0,
        }
    ),
    RFModelConfig(
        model_name="Qwen/Qwen2.5-1.5B-Instruct",
        peft_config=peft_configs,
        training_args=make_sft_args(2e-4),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=fmt_func,
        compute_metrics=compute_metrics_for_rf,
        generation_config={
            "max_new_tokens": 256,
            "do_sample": False,
            "temperature": 1.0,
        }
    ),
])

print(f"Config set defined: {len(config_set.values)} base configs × 2 LoRA adapters = 8 runs")

## 9. Initialize Experiment and Launch Grid Search

In [ ]:
# Every experiment instance must be uniquely named — mirrors tutorial pattern
experiment = Experiment(
    experiment_name="proj2-schema-linking-v1",
    mode="fit",
    experiment_path="./logs"   # RapidFire saves rapidfire.log + per-run metrics here
)

In [ ]:
# Build the config group via grid search — all 8 combinations run concurrently
# num_chunks controls swap granularity (how often IC Ops can redirect compute)
config_group = RFGridSearch(
    configs=config_set,
    trainer_type="SFT"
)

In [ ]:
cuda_cleanup()

In [ ]:
# Launch training
# Tip: monitor the logs/ directory for live metrics; use experiment.stop(run_id) to
# kill underperforming runs early and redirect GPU budget to better configs.
experiment.run_fit(
    config_group,
    create_model,
    train_ds,
    val_ds,
    num_chunks=4,   # finer granularity for 8 configs new! switched to 4
    seed=42
)

## 10. Inspect Results & Pick Best Config

In [ ]:
# Get a summary table of all runs — mirrors what the tutorial calls get_results()
results = experiment.get_results()
print(results)

# Get per-run metadata (model name, LoRA rank, LR, best metric, etc.)
runs_info = experiment.get_runs_info()
for run in runs_info:
    print(run)

In [ ]:
# ---- Plot leaderboard scores across configs --------------------------------
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    df = pd.DataFrame(runs_info)
    df_sorted = df.sort_values("leaderboard_score", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(df_sorted["run_id"].astype(str), df_sorted["leaderboard_score"], color="steelblue")
    ax.axvline(0.5, color="red",    linestyle="--", label="Target ≥0.50")
    ax.axvline(0.6, color="orange", linestyle="--", label="Good ≥0.60")
    ax.set_xlabel("Leaderboard Score"); ax.set_title("Multi-Config Sweep — Schema Linking")
    ax.legend(); plt.tight_layout(); plt.savefig("logs/config_comparison.png", dpi=120)
    plt.show()
except ImportError:
    print("matplotlib/pandas not installed — skip plot")

## 11. Interactive Control Operations (IC Ops)

Use these cells to redirect compute mid-sweep — stop underperformers and clone-modify the best config to explore nearby hyperparameters.

In [ ]:
# ---- Stop a specific underperforming run (replace with actual run_id) ------
# experiment.stop(run_id="run_id_here")

# ---- Resume a paused run ---------------------------------------------------
# experiment.resume(run_id="run_id_here")

# ---- Clone-and-modify best run with a new LR --------------------------------
# best_run_id = runs_info[0]["run_id"]   # after sorting by leaderboard_score
# experiment.clone_modify(
#     run_id=best_run_id,
#     modifications={"training_args": {"learning_rate": 5e-5}}
# )

## 12. Post-Processing — JSON Repair & Identifier Validation

**Critical for score:** identifiers absent from the schema count as false positives (hallucinations), directly lowering precision. A post-processing pass that:
1. Parses/repairs malformed JSON from the model output, and
2. Drops any table/column names not present in the schema

is often worth more than an extra epoch of training.

In [ ]:
def postprocess_output(raw_text: str, db_id: str) -> dict:
    """
    Parse model output and filter to valid schema identifiers only.
    Returns a schema_links dict: {table_name: [col_name, ...]}.
    Uses case-insensitive matching but preserves original schema casing in output.
    """
    schema = load_schema(db_id)
    # Build lookup: lower_table -> original_table, lower_col -> original_col per table
    table_lower2orig = {t.lower(): t for t in schema["table_names_original"]}
    col_lower2orig   = {}
    for (t_idx, cname) in schema["column_names_original"]:
        tname = schema["table_names_original"][t_idx]
        col_lower2orig.setdefault(tname.lower(), {})[cname.lower()] = cname

    pred = _parse_json_output(raw_text)
    # if not pred:
    #     # Fallback: return all tables with empty column lists (maximises recall)
    #     return {t: [] for t in schema["table_names_original"]}
    if not pred:
        return {}   # let the scorer give 0 rather than polluting precision

    validated = {}
    for pred_table, pred_cols in pred.items():
        t_lower = pred_table.lower()
        if t_lower not in table_lower2orig:
            continue  # hallucinated table — drop
        orig_table = table_lower2orig[t_lower]
        valid_cols = []
        for c in (pred_cols or []):
            c_lower = c.lower()
            col_map  = col_lower2orig.get(t_lower, {})
            if c_lower in col_map:
                valid_cols.append(col_map[c_lower])  # restore original casing
            # else: hallucinated column — silently drop
        validated[orig_table] = valid_cols

    return validated


# Quick test
sample_output = '{"orders": ["orderid", "fakecol"], "CUSTOMERS": []}'
cleaned = postprocess_output(sample_output, raw_val[0]["db_id"])
print("Post-processed:", cleaned)

## 13. `main.py` Inference Script

The cell below writes the full `main.py` that the TAs will run. It loads the best adapter from `./adapter/`, runs batched inference, applies post-processing, and writes the output JSON.

**Invocation:** `python3 main.py --input validation_input.json --output predictions.json`

In [ ]:
MAIN_PY = '''#!/usr/bin/env python3
"""
main.py — Schema-linking inference script for CSE/DSC 234 Project 2.

Usage:
    python3 main.py --input <input_json> --output <output_json>
                    [--batch_size 8] [--checkpoint_path ./adapter]
                    [--model_name Qwen/Qwen2.5-1.5B-Instruct]
"""
import argparse, json, re, os
from pathlib import Path

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--input",           required=True)
    p.add_argument("--output",          required=True)
    p.add_argument("--batch_size",      type=int, default=8)
    p.add_argument("--checkpoint_path", default="./adapter")
    p.add_argument("--model_name",      default="Qwen/Qwen2.5-1.5B-Instruct")
    return p.parse_args()

# ---- Schema helpers (same as training notebook) ----------------------------
_schema_cache = {}
def load_schema(db_id, schemas_dir="./schemas"):
    if db_id not in _schema_cache:
        fname = db_id.replace(" ", "_") + ".json"
        with open(Path(schemas_dir) / fname) as f:
            _schema_cache[db_id] = json.load(f)
    return _schema_cache[db_id]

def serialize_schema_minimal(schema):
    lines = []
    for t_idx, tname in enumerate(schema["table_names_original"]):
        cols = [c for (ti, c) in schema["column_names_original"] if ti == t_idx]
        lines.append(f"{tname}({\', \'.join(cols)})")
    return "\\n".join(lines)

def _parse_json_output(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\\s*", "", text)
    text = re.sub(r"\\s*```$",         "", text)
    m = re.search(r"\\{.*\\}", text, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except json.JSONDecodeError: pass
    return {}

def postprocess_output(raw_text, db_id):
    schema = load_schema(db_id)
    table_lower2orig = {t.lower(): t for t in schema["table_names_original"]}
    col_lower2orig = {}
    for (t_idx, cname) in schema["column_names_original"]:
        tname = schema["table_names_original"][t_idx]
        col_lower2orig.setdefault(tname.lower(), {})[cname.lower()] = cname
    pred = _parse_json_output(raw_text)
    if not pred:
        return {t: [] for t in schema["table_names_original"]}
    validated = {}
    for pred_table, pred_cols in pred.items():
        t_lower = pred_table.lower()
        if t_lower not in table_lower2orig:
            continue
        orig_table = table_lower2orig[t_lower]
        valid_cols = []
        for c in (pred_cols or []):
            col_map = col_lower2orig.get(t_lower, {})
            if c.lower() in col_map:
                valid_cols.append(col_map[c.lower()])
        validated[orig_table] = valid_cols
    return validated

SYSTEM_PROMPT = (
    "You are a schema-linking assistant. Given a natural-language question and a database schema, "
    "output ONLY a JSON object mapping each referenced table name to the list of referenced column names. "
    "If a table is referenced but no specific columns are needed (e.g. SELECT COUNT(*)), map it to []. "
    "Do not output any explanation, markdown, or extra text — only the JSON object."
)

def build_prompt(question, db_id):
    schema   = load_schema(db_id)
    schema_s = serialize_schema_minimal(schema)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Database schema:\\n{schema_s}\\n\\nQuestion: {question}"},
    ]

# ---- Main ------------------------------------------------------------------
def main():
    args = parse_args()

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    print(f"Loading base model: {args.model_name}")
    base = AutoModelForCausalLM.from_pretrained(
        args.model_name, device_map="auto", torch_dtype="auto", use_cache=True
    )
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading LoRA adapter from: {args.checkpoint_path}")
    model = PeftModel.from_pretrained(base, args.checkpoint_path)
    model.eval()

    with open(args.input) as f:
        inputs = json.load(f)

    outputs = []
    batch_size = args.batch_size

    for i in range(0, len(inputs), batch_size):
        batch = inputs[i : i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                build_prompt(ex["question"], ex["db_id"]),
                tokenize=False, add_generation_prompt=True
            )
            for ex in batch
        ]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=1024).to(model.device)
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        for ex, input_ids_row, gen_ids_row in zip(batch, enc["input_ids"], gen):
            new_tokens = gen_ids_row[len(input_ids_row):]
            raw_text   = tokenizer.decode(new_tokens, skip_special_tokens=True)
            schema_links = postprocess_output(raw_text, ex["db_id"])
            outputs.append({"question_id": ex["question_id"], "schema_links": schema_links})
        print(f"  Processed {min(i + batch_size, len(inputs))}/{len(inputs)} questions")

    with open(args.output, "w") as f:
        json.dump(outputs, f, indent=2)
    print(f"Saved {len(outputs)} predictions to {args.output}")

if __name__ == "__main__":
    main()
'''

with open("main.py", "w") as f:
    f.write(MAIN_PY)
print("main.py written — run:  python3 main.py --input validation_input.json --output predictions.json")

## 14. End Experiment

In [ ]:
experiment.end()

---

### Schema-Style Ablation (Optional Next Step)

Re-run the sweep with `SCHEMA_STYLE = "full"` to measure the effect of richer schema context
(types + PK/FK annotations). Compare leaderboard scores to the `minimal` run.

### Augmentation Ideas

- **Paraphrase questions** via an offline LLM call (document in report).
- **Synthesise NL–SQL pairs** for under-represented `db_id` values (SBODemoUS sub-modules),
  then extract schema links with `sql_to_schema_links.py`.
- **Balance the training set** by upsampling schemas with fewer than 10 examples.